In [85]:
import pandas as pd
import numpy as np

# Load discretized dataset
df = pd.read_csv("D:\\Coding\\6th sem\\Soft Computing\\Credit Risk Assessment\\data\\Rough_Decoded_german_data.csv",keep_default_na=False)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (1000, 21)
   CheckingAccountStatus Duration               CreditHistory  \
0                      1    Short            Critical account   
1                      1    Short            Critical account   
2                      2    Short  Existing credits paid back   
3                      1    Short            Critical account   
4                      1    Short            Critical account   

               Purpose CreditAmount        SavingsAccount EmploymentDuration  \
0  Furniture/equipment          Low              < 100 DM           < 1 year   
1            Car (new)          Low              < 100 DM        1 - 4 years   
2             Business          Low  100 <= savings < 500        4 - 7 years   
3            Car (new)          Low              < 100 DM        1 - 4 years   
4            Car (new)          Low              < 100 DM        1 - 4 years   

   InstallmentRate                     PersonalStatus Guarantor  ...  \
0                4  Female div

In [86]:
print(df["Risk"].unique())
print(df["Risk"].value_counts())


['Low Risk' 'High Risk']
Risk
Low Risk     700
High Risk    300
Name: count, dtype: int64


In [87]:
selected_cols = [
    "CheckingAccountStatus",
    "CreditHistory",
    "SavingsAccount",
    "Duration",
    "CreditAmount",
    "Age"
]


In [88]:
indiscernibility = df.groupby(selected_cols).groups

group_sizes = [len(v) for v in indiscernibility.values()]

print("Average group size:", sum(group_sizes)/len(group_sizes))
print("Max group size:", max(group_sizes))


Average group size: 3.4722222222222223
Max group size: 65


In [89]:
#Lower and Upper approximation
selected_col = "Risk"
target_class = "High Risk"

target_objects = set(df[df[selected_col] == target_class].index)

lower_approx = set()
upper_approx = set()

for group in indiscernibility.values():
    group_set = set(group)

    if group_set.issubset(target_objects):
        lower_approx.update(group_set)

    if group_set & target_objects:
        upper_approx.update(group_set)

boundary_region = upper_approx - lower_approx

print("Lower Approximation:", len(lower_approx))
print("Upper Approximation:", len(upper_approx))
print("Boundary Region:", len(boundary_region))


Lower Approximation: 80
Upper Approximation: 741
Boundary Region: 661


In [90]:
group_sizes = [len(v) for v in indiscernibility.values()]
print("Average group size:", sum(group_sizes)/len(group_sizes))
print("Min group size:", min(group_sizes))
print("Max group size:", max(group_sizes))


Average group size: 3.4722222222222223
Min group size: 1
Max group size: 65


In [91]:
#accuracy
accuracy = len(lower_approx) / len(upper_approx)
print("Approximation Accuracy:", round(accuracy, 4))

Approximation Accuracy: 0.108


In [92]:
def dependency_degree(attributes):
    grouped = df.groupby(attributes)
    consistent_objects = 0

    for _, group in grouped:
        # If all objects in this equivalence class
        # belong to the same decision class
        if len(group["Risk"].unique()) == 1:
            consistent_objects += len(group)

    return consistent_objects / len(df)


In [93]:
print("Dependency Degree:",
      round(dependency_degree(selected_cols), 4))


Dependency Degree: 0.339


In [94]:
df.to_csv("D:\\Coding\\6th sem\\Soft Computing\\Credit Risk Assessment\\data\\Final_Rough_Decoded_german_data.csv", index=False)


In [95]:
with open("D:\\Coding\\6th sem\\Soft Computing\\Credit Risk Assessment\\data\\RST_metadata.txt", "w") as f:
    f.write("Selected Columns:\n")
    for col in selected_cols:
        f.write(col + "\n")

    f.write("\nDependency Degree: 0.339\n")
    f.write("Approximation Accuracy: 0.108\n")
    f.write("Lower Approximation: 80\n")
    f.write("Upper Approximation: 741\n")
    f.write("Boundary Region: 661\n")


In [96]:
#decision Rules 


In [97]:
lower_df = df.loc[list(lower_approx)]


In [98]:
decision_col = "Risk"
condition_cols = [col for col in lower_df.columns if col != decision_col]

grouped = lower_df.groupby(condition_cols)

rules = []

for name, group in grouped:
    
    decision_value = group[decision_col].iloc[0]
    
    rule_conditions = dict(zip(condition_cols, name))
    
    rules.append((rule_conditions, decision_value))


In [99]:
for i, (conditions, decision) in enumerate(rules, 1):
    
    cond_str = " AND ".join(
        [f"{k} = {v}" for k, v in conditions.items()]
    )
    
    print(f"Rule {i}:")
    print(f"IF {cond_str}")
    print(f"THEN {decision_col} = {decision}\n")


Rule 1:
IF CheckingAccountStatus = 1 AND Duration = Long AND CreditHistory = Delay in paying AND Purpose = Business AND CreditAmount = Medium AND SavingsAccount = < 100 DM AND EmploymentDuration = >= 7 years AND InstallmentRate = 3 AND PersonalStatus = Male single AND Guarantor =  AND ResidenceDuration = 4 AND Property = Unknown / None AND Age = Old AND OtherInstallmentPlans =  AND Housing = Own AND ExistingCredits = 2 AND Job = Skilled AND Dependents = 2 AND Telephone = Yes AND ForeignWorker = No
THEN Risk = High Risk

Rule 2:
IF CheckingAccountStatus = 1 AND Duration = Long AND CreditHistory = Existing credits paid back AND Purpose = Business AND CreditAmount = Medium AND SavingsAccount = < 100 DM AND EmploymentDuration = >= 7 years AND InstallmentRate = 4 AND PersonalStatus = Male single AND Guarantor = Co-applicant AND ResidenceDuration = 4 AND Property = Unknown / None AND Age = Young AND OtherInstallmentPlans =  AND Housing = Rent AND ExistingCredits = 1 AND Job = Skilled AND Dep

In [100]:
#redusction
decision_col = "Risk"

condition_cols = selected_cols  # use your 6 features only

grouped = df.groupby(condition_cols)

target_class = "High Risk"

lower_approx = set()
upper_approx = set()

for name, group in grouped:
    
    indices = set(group.index)
    
    # Upper approximation
    if target_class in group[decision_col].values:
        upper_approx.update(indices)
    
    # Lower approximation (certain region)
    if len(group[decision_col].unique()) == 1 and group[decision_col].iloc[0] == target_class:
        lower_approx.update(indices)

print("Lower:", len(lower_approx))
print("Upper:", len(upper_approx))
print("Boundary:", len(upper_approx - lower_approx))


Lower: 80
Upper: 741
Boundary: 661


In [101]:
lower_df = df.loc[list(lower_approx)]

grouped_lower = lower_df.groupby(condition_cols)

rules = []

for name, group in grouped_lower:
    
    decision_value = group[decision_col].iloc[0]
    rule_conditions = dict(zip(condition_cols, name))
    
    rules.append((rule_conditions, decision_value))


In [102]:
for i, (conditions, decision) in enumerate(rules, 1):
    
    cond_str = " AND ".join(
        [f"{k} = {v}" for k, v in conditions.items()]
    )
    
    print(f"Rule {i}:")
    print(f"IF {cond_str}")
    print(f"THEN Risk = {decision}\n")


Rule 1:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Medium AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk

Rule 2:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Medium AND CreditAmount = Medium AND Age = Young
THEN Risk = High Risk

Rule 3:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Medium AND Age = Middle
THEN Risk = High Risk

Rule 4:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = Unknown / None AND Duration = Short AND CreditAmount = Low AND Age = Middle
THEN Risk = High Risk

Rule 5:
IF CheckingAccountStatus = 1 AND CreditHistory = Critical account AND SavingsAccount = 500 <= savings < 1000 AND Duration = Medium AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk

Rule 6:
IF Chec

In [103]:
min_support = 5

for name, group in grouped_lower:
    
    if len(group) >= min_support:
        
        decision_value = group[decision_col].iloc[0]
        rule_conditions = dict(zip(condition_cols, name))
        
        rules.append((rule_conditions, decision_value))


In [104]:
for i, (conditions, decision) in enumerate(rules, 1):
    
    cond_str = " AND ".join(
        [f"{k} = {v}" for k, v in conditions.items()]
    )
    
    print(f"Rule {i}:")
    print(f"IF {cond_str}")
    print(f"THEN Risk = {decision}\n")


Rule 1:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Medium AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk

Rule 2:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Medium AND CreditAmount = Medium AND Age = Young
THEN Risk = High Risk

Rule 3:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Medium AND Age = Middle
THEN Risk = High Risk

Rule 4:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = Unknown / None AND Duration = Short AND CreditAmount = Low AND Age = Middle
THEN Risk = High Risk

Rule 5:
IF CheckingAccountStatus = 1 AND CreditHistory = Critical account AND SavingsAccount = 500 <= savings < 1000 AND Duration = Medium AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk

Rule 6:
IF Chec

In [105]:
#support for each rule
rules = []

for name, group in df.groupby(selected_cols):
    
    total = len(group)
    high_risk_count = len(group[group["Risk"] == "High Risk"])
    
    confidence = high_risk_count / total
    
    if confidence >= 0.8 and total >= 10:
        
        rule_conditions = dict(zip(selected_cols, name))
        
        rules.append((rule_conditions, total, confidence))


In [106]:
for i, (conditions, support, confidence) in enumerate(rules, 1):
    
    cond_str = " AND ".join(
        [f"{k} = {v}" for k, v in conditions.items()]
    )
    
    print(f"Rule {i}:")
    print(f"IF {cond_str}")
    print("THEN Risk = High Risk")
    print(f"Support = {support}")
    print(f"Confidence = {round(confidence, 2)}\n")


Rule 1:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk
Support = 10
Confidence = 0.8



In [107]:
#slightly lower confidence 
rules = []

min_support = 8
min_confidence = 0.75

for name, group in df.groupby(selected_cols):
    
    total = len(group)
    high_risk_count = len(group[group["Risk"] == "High Risk"])
    
    if total > 0:
        confidence = high_risk_count / total
        
        if confidence >= min_confidence and total >= min_support:
            
            rule_conditions = dict(zip(selected_cols, name))
            
            rules.append((rule_conditions, total, confidence))


In [108]:
for i, (conditions, support, confidence) in enumerate(rules, 1):
    
    cond_str = " AND ".join(
        [f"{k} = {v}" for k, v in conditions.items()]
    )
    
    print(f"Rule {i}:")
    print(f"IF {cond_str}")
    print("THEN Risk = High Risk")
    print(f"Support = {support}")
    print(f"Confidence = {round(confidence, 2)}\n")


Rule 1:
IF CheckingAccountStatus = 1 AND CreditHistory = All credits paid back AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk
Support = 10
Confidence = 0.8

Rule 2:
IF CheckingAccountStatus = 2 AND CreditHistory = Critical account AND SavingsAccount = < 100 DM AND Duration = Medium AND CreditAmount = Low AND Age = Young
THEN Risk = High Risk
Support = 8
Confidence = 0.75



In [109]:
#FOR LOW RISK
low_rules = []

min_support = 8
min_confidence = 0.75

for name, group in df.groupby(selected_cols):
    
    total = len(group)
    low_risk_count = len(group[group["Risk"] == "Low Risk"])
    
    if total > 0:
        confidence = low_risk_count / total
        
        if confidence >= min_confidence and total >= min_support:
            
            rule_conditions = dict(zip(selected_cols, name))
            
            low_rules.append((rule_conditions, total, confidence))


In [110]:
for i, (conditions, support, confidence) in enumerate(low_rules, 1):
    
    cond_str = " AND ".join(
        [f"{k} = {v}" for k, v in conditions.items()]
    )
    
    print(f"Low Risk Rule {i}:")
    print(f"IF {cond_str}")
    print("THEN Risk = Low Risk")
    print(f"Support = {support}")
    print(f"Confidence = {round(confidence, 2)}\n")


Low Risk Rule 1:
IF CheckingAccountStatus = 1 AND CreditHistory = Critical account AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Low AND Age = Middle
THEN Risk = Low Risk
Support = 16
Confidence = 0.75

Low Risk Rule 2:
IF CheckingAccountStatus = 1 AND CreditHistory = Critical account AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Low AND Age = Young
THEN Risk = Low Risk
Support = 21
Confidence = 0.76

Low Risk Rule 3:
IF CheckingAccountStatus = 3 AND CreditHistory = Existing credits paid back AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Low AND Age = Young
THEN Risk = Low Risk
Support = 17
Confidence = 0.76

Low Risk Rule 4:
IF CheckingAccountStatus = 4 AND CreditHistory = Critical account AND SavingsAccount = < 100 DM AND Duration = Short AND CreditAmount = Low AND Age = Middle
THEN Risk = Low Risk
Support = 20
Confidence = 0.95

Low Risk Rule 5:
IF CheckingAccountStatus = 4 AND CreditHistory = Critical account A

In [111]:
#Compute RULE COVERAGE
high_coverage = sum([support for _, support, _ in rules])

total_samples = len(df)

high_coverage_percent = (high_coverage / total_samples) * 100

print("High Risk Rule Coverage:", round(high_coverage_percent, 2), "%")


High Risk Rule Coverage: 1.8 %


In [112]:
low_coverage = sum([support for _, support, _ in low_rules])

low_coverage_percent = (low_coverage / total_samples) * 100

print("Low Risk Rule Coverage:", round(low_coverage_percent, 2), "%")


Low Risk Rule Coverage: 27.2 %


In [113]:
#total coverage
total_rule_coverage = high_coverage + low_coverage

total_rule_coverage_percent = (total_rule_coverage / total_samples) * 100

print("Total Rule Coverage:", round(total_rule_coverage_percent, 2), "%")


Total Rule Coverage: 29.0 %


In [114]:
#SAVING
df.to_csv("D:\\Coding\\6th sem\\Soft Computing\\Credit Risk Assessment\\data\\CSV_after_rule_extraction.csv", index=False)


In [115]:
print(df.shape)
print(df.columns)
print(df.head())



(1000, 21)
Index(['CheckingAccountStatus', 'Duration', 'CreditHistory', 'Purpose',
       'CreditAmount', 'SavingsAccount', 'EmploymentDuration',
       'InstallmentRate', 'PersonalStatus', 'Guarantor', 'ResidenceDuration',
       'Property', 'Age', 'OtherInstallmentPlans', 'Housing',
       'ExistingCredits', 'Job', 'Dependents', 'Telephone', 'ForeignWorker',
       'Risk'],
      dtype='object')
   CheckingAccountStatus Duration               CreditHistory  \
0                      1    Short            Critical account   
1                      1    Short            Critical account   
2                      2    Short  Existing credits paid back   
3                      1    Short            Critical account   
4                      1    Short            Critical account   

               Purpose CreditAmount        SavingsAccount EmploymentDuration  \
0  Furniture/equipment          Low              < 100 DM           < 1 year   
1            Car (new)          Low             

In [116]:
#saving the rules

In [117]:
import json

high_rules_json = []
low_rules_json = []

# High Risk rules
high_rules_json.append({
    "conditions": {
        "CheckingAccountStatus": 1,
        "CreditHistory": "All credits paid back",
        "SavingsAccount": "< 100 DM",
        "Duration": "Short",
        "CreditAmount": "Low",
        "Age": "Young"
    },
    "support": 10,
    "confidence": 0.8
})

high_rules_json.append({
    "conditions": {
        "CheckingAccountStatus": 2,
        "CreditHistory": "Critical account",
        "SavingsAccount": "< 100 DM",
        "Duration": "Medium",
        "CreditAmount": "Low",
        "Age": "Young"
    },
    "support": 8,
    "confidence": 0.75
})

# Low Risk rules (example — add all 14 similarly)
low_rules_json.append({
    "conditions": {
        "CheckingAccountStatus": 4,
        "CreditHistory": "Existing credits paid back",
        "SavingsAccount": "< 100 DM",
        "Duration": "Short",
        "CreditAmount": "Low",
        "Age": "Young"
    },
    "support": 50,
    "confidence": 0.88
})

# Combine
all_rules = {
    "high_risk_rules": high_rules_json,
    "low_risk_rules": low_rules_json
}

with open(r"D:\\Coding\\6th sem\\Soft Computing\\Credit Risk Assessment\\rules\\rules.json", "w") as f:
    json.dump(all_rules, f, indent=4)
